In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [12]:
load_dotenv()

NEBULA_BASE_URL = 'https://nebula.cs.vu.nl/api/'
NEBULA_API_KEY = os.getenv('NEBULA_API_KEY')
NEBULA = OpenAI(base_url=NEBULA_BASE_URL, api_key=NEBULA_API_KEY)
MODEL = "FAST.llama3.2-vision:11b"

### Available Model in Nebula

In [4]:
def get_nebula_models():
    models = []
    for model in NEBULA.models.list().data:
        models.append(model.id)
    return models

In [5]:
available_models = get_nebula_models()
print(f"Models: {available_models}\n\n")

Models: ['deepseek-r1:8b', 'deepseek-r1:1.5b', 'FAST.gemma3:12b', 'FAST.gpt-oss:120b', 'FAST.gpt-oss:20b', 'FAST.llama3.2-vision:11b', 'FAST.qwen3-vl:8b', 'go-assist', 'research-railway-guide', 'vu-rdm-support-chatbot---test', 'pluto.gemma3:12b', 'mercury.gpt-oss:20b', 'pluto.llama3.1:8b']




In [ ]:
def prompt_nebula(model, system_prompt, user_prompt, configs=None):
    prompt_parameters = {
        "model": model,
        "messages": [
            { "role": "system", "content": system_prompt },
            { "role": "user", "content": user_prompt }
        ],
    }

    if configs:
        prompt_parameters.update(configs)

    response = NEBULA.chat.completions.create(**prompt_parameters)

    return response

if __name__=='__main__':
    available_models = get_nebula_models()
    print(f"Models: {available_models}\n\n")
    
    config = {
        "max_tokens": 25
    }

    response = prompt_nebula(MODEL, "You are a helpful assistant", "What is the capital of Brazil?", config)
    print(f"Response: {response.choices[0].message.content}\n\n")
    print(f"Usage stats: {response.usage}")

### Multi-turn conversation example below ###

In [7]:
import base64
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
from prompt import cell_detection_prompt, htr_prompt, reconstruct_table_prompt

def call_LLM(image_path, model_name=MODEL, temperature=0):
    base64_image = encode_image(image_path)
    prompt= "Here is a table image. Please collaborate to: (1) detect table cells, (2) run HTR, (3) reconstruct the HTML table."
    content = [{"type": "text", "text": prompt}]

    content.append({
       "type": "image_url",
       "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}",
        }
    })

    response = NEBULA.chat.completions.create(
       model=model_name,
        messages=[
            {
                "role": "user",
                "content": content,
            },
            {
                "role": "user",
                "content": f"{cell_detection_prompt}"
            },
            {
                "role": "user",
                "content": f"{htr_prompt}"
            },
            {
                "role": "user",
                "content": f"{reconstruct_table_prompt}"
            }
        ],
        temperature=temperature,
    )

    result = response.choices[0].message.content
    print(result)
    return result




In [ ]:
image_path = "../data/images/NL-HaNA_2.10.50_45_0131.jpg"
if __name__ == "__main__":
   call_LLM(image_path)

### Example call to Nebula LLM with an image

In [8]:
import base64
from PIL import Image
import io

def print_resolution(img, label):
    print(f"{label} resolution: {img.width} x {img.height}")

def main():
    input_image_path = "../data/images/NL-HaNA_2.10.50_45_0110.jpg"
    output_image_path = "output.jpg"

    # --- Step 1: Read Original Image ---
    original_img = Image.open(input_image_path)
    print_resolution(original_img, "Original")

    # --- Step 2: Convert image to Base64 ---
    with open(input_image_path, "rb") as img_file:
        img_bytes = img_file.read()
        img_base64 = base64.b64encode(img_bytes)

    print("Base64 length:", len(img_base64))

    # --- Step 3: Convert Base64 back to Image ---
    decoded_bytes = base64.b64decode(img_base64)
    decoded_img = Image.open(io.BytesIO(decoded_bytes))
    print_resolution(decoded_img, "Decoded")

    # --- Step 4: Save decoded image ---
    decoded_img.save(output_image_path)
    print(f"Decoded image saved as {output_image_path}")

    # --- Step 5: Verify images are identical ---
    print("\nVerifying if original and decoded images are identical...")

    # comparing raw bytes exactly
    with open(input_image_path, "rb") as f1, open(output_image_path, "rb") as f2:
        identical = f1.read() == f2.read()

    print("Images are identical:", identical)

if __name__ == "__main__":
    main()


Original resolution: 5000 x 3233
Base64 length: 1334788
Decoded resolution: 5000 x 3233
Decoded image saved as output.jpg

Verifying if original and decoded images are identical...
Images are identical: False


In [9]:
def prompt_nebula(model, system_prompt, user_prompt, configs=None):
    prompt_parameters = {
        "model": model,
        "messages": [
            { "role": "system", "content": system_prompt },
            { "role": "user", "content": user_prompt }
        ],
    }

    if configs:
        prompt_parameters.update(configs)

    response = NEBULA.chat.completions.create(**prompt_parameters)

    return response

In [13]:
from prompt import tsr_html_prompt
def call_LLM(image_path, prompt=tsr_html_prompt, model_name=MODEL, temperature=0):
    base64_image = encode_image(image_path)
    content = [{"type": "text", "text": prompt}]

    content.append({
       "type": "image_url",
       "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}",
        }
    })

    response = NEBULA.chat.completions.create(
       model=model_name,
        messages=[
            {
                "role": "user",
                "content": content,
            }
        ],
        temperature=temperature,
    )

    result = response.choices[0].message.content
    print(result)
    return result

In [15]:
image_path = "../data/images/NL-HaNA_2.10.50_45_0131.jpg"
if __name__ == "__main__":
   call_LLM(image_path)

Here is the table structure with content in HTML format:

<table>
  <thead>
    <tr>
      <th>Namen en Toenamen.</th>
      <th>Namen der Ouders, datum van Geboorte, Geboorteplaats en huw. Woonplaats.</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
    </tr>
    <tr>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
    </tr>
    <tr>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
    </tr>
    <tr>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
      <td>van Huisbroek, 2. Januari 1823, Geerstap, 1. Februari 1851, Geerstap.</td>
    </tr>
    <t